In [9]:
# 库引用部分
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import clear_output, display
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.losses import Loss
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
class R2PlotCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        self.train_r2 = []
        self.val_r2 = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.train_r2.append(logs.get("r_squared", 0))
        self.val_r2.append(logs.get("val_r_squared", 0))

        # 实时更新图像
        clear_output(wait=True)  # 清除前一张图
        plt.figure(figsize=(8, 6))
        plt.plot(
            self.train_r2, label="Train R2", marker="o", linestyle="-", markersize=1
        )
        plt.plot(self.val_r2, label="Test R2", marker="s", linestyle="-", markersize=1)
        plt.xlabel("Epochs")
        plt.ylabel("R2 Score")
        plt.title("Training and Validation R2 Over Epochs")
        plt.legend()
        plt.grid(True)
        plt.show()


r2_plot_callback = R2PlotCallback()
# 数据引入部分
X = pd.read_csv("curves_4points.csv")
y = pd.read_csv("shuchu10.csv")
# print(X)
# X = X.values.reshape(-1, 16)  # 每条曲线被展平成16维的向量 (8*2)
# print(X)
import pandas as pd
from sklearn.preprocessing import RobustScaler
import joblib

# # 读取原始输出数据
# y = pd.read_csv("shuchu.csv").values

from sklearn.preprocessing import MinMaxScaler, RobustScaler
import joblib

# 训练时拟合
X_scaler = MinMaxScaler()
X_scaled = X_scaler.fit_transform(X)
joblib.dump(X_scaler, "X_scaler.save")

Y_scaler = RobustScaler()
Y_scaled = Y_scaler.fit_transform(y)
joblib.dump(Y_scaler, "Y_scaler.save")


# 数据标准化
scaler_X_minmax = MinMaxScaler(feature_range=(0, 1))
X = scaler_X_minmax.fit_transform(X)

scaler_y = RobustScaler()
# 拟合并保存 scaler
y_scaler = RobustScaler()
y_scaler.fit(y)
joblib.dump(y_scaler, "y_scaler.save")
print("RobustScaler 已保存为 y_scaler.save")

y = scaler_y.fit_transform(y)


X_else=X[361:367]                  #已归一化后的需反演X值
X=X[0:361]
print('X:',X,'X_else:',X_else)
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=43)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.1, random_state=43
# )
print("X_train min:", X_test.min(), "max:", X_test.max())
print("y_scaled min:", y_test.min(), "max:", y_test.max())
# mean=X_train.mean(axis=0)          #axis=0 表示沿着每一列进行操作，也就是求出每一列的均值
# X_train-=mean                      #每列数据以该列均值为中心
# std=X_train.std(axis=0)
# X_train/=std

# X_test-=mean
# X_test/=std

# mean=y_train.mean(axis=0)          #axis=0 表示沿着每一列进行操作，也就是求出每一列的均值
# y_train-=mean                      #每列数据以该列均值为中心
# std=y_train.std(axis=0)
# y_train/=std

# y_test-=mean
# y_test/=std


# 自定义损失函数部分
def r_squared(y_true, y_pred):
    ss_total = tf.reduce_sum(tf.square(y_true - tf.reduce_mean(y_true)))
    ss_residual = tf.reduce_sum(tf.square(y_true - y_pred))
    r2 = 1 - (ss_residual / ss_total)
    return r2  # 返回R²值


class WeightedMSE(Loss):
    def __init__(self, weights, name="weighted_mse"):
        super().__init__(name=name)
        self.weights = tf.constant(weights, dtype=tf.float32)

    def call(self, y_true, y_pred):
        mse = tf.square(y_true - y_pred)
        weighted_mse = mse * self.weights
        return tf.reduce_mean(weighted_mse)


# 每个维度设置不同的权重：
output_weights = [0.01, 0.1, 0.89]
# output_weights = [0.01, 0.1, 0.1,0.1,0.2,0.2,0.09]
weighted_mse_loss = WeightedMSE(output_weights)


# 模型构建
def get_model():
    model = Sequential()
    model.add(Dense(64, input_dim=X.shape[1], activation="relu"))
    model.add(Dense(124, activation="relu"))
    # model.add(Dropout(0.5))
    # model.add(Dropout(0.15))
    model.add(Dense(64, activation="relu"))
    model.add(Dense(32,activation="relu"))
    model.add(Dense(3))
    model.compile(
        optimizer=Adam(learning_rate=0.0007),
        # optimizer=SGD(learning_rate=0.1),
        loss=weighted_mse_loss,
        metrics=[r_squared],
    )
    return model


# 这是不用K折验证做的

# 设置回调函数
# checkpoint = ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_loss', mode='min')
model = get_model()
history = model.fit(
    X_train,
    y_train,
    epochs=1000,
    batch_size=32,
    verbose=1,
    validation_data=(X_test, y_test),
    # callbacks=[r2_plot_callback],
    # callbacks=[checkpoint],
)




model.save("ANN_model.keras")
# 获取训练集和验证集的损失和 R-squared 值
train_loss = history.history["loss"]
val_loss = history.history["val_loss"]
train_r_squared = history.history["r_squared"]  # 训练集的 R-squared
val_r_squared = history.history["val_r_squared"]  # 验证集的 R-squared

# 获取最低的损失值
min_train_loss = min(train_loss)
min_val_loss = min(val_loss)

# 获取最高和最低的 R-squared 值
max_train_r_squared = max(train_r_squared)
max_val_r_squared = max(val_r_squared)

# 输出结果
print("Max Train R-squared:", max_train_r_squared)
print("Min Train Loss:", min_train_loss)
print("Max Val R-squared:", max_val_r_squared)
print("Min Val Loss:", min_val_loss)





# y_pred = model.predict(X_test)
# y_pred1 =model.predict(X_train)
# result=model.predict(X_else)
# print('反归一化之前预测结果：', result)


# result_inv = scaler_y.inverse_transform(result)
# print('归一化之后预测结果:', result_inv)
# # # === 1. 正确拼接 y_test 和 y_pred 为二维数组 ===这是验证集
# # y_test_concat = np.concatenate([y.reshape(-1, 1) for y in y_test], axis=1)
# # y_pred_concat = np.concatenate([y.reshape(-1, 1) for y in y_pred], axis=1)

# # === 2. 反归一化 ===验证集反归一化
# # y_test_inv = scaler_X_minmax.inverse_transform(y_test_concat)
# # y_pred_inv = scaler_X_minmax.inverse_transform(y_pred_concat)
# y_test_inv = scaler_y.inverse_transform(y_test)
# y_pred_inv = scaler_y.inverse_transform(y_pred)

# # # === 3. 正确拼接 y_train 和 y_pred1 为二维数组 ===这是训练集
# # y_test_concat1 = np.concatenate([y.reshape(-1, 1) for y in y_train], axis=1)
# # y_pred_concat1 = np.concatenate([y.reshape(-1, 1) for y in y_pred1], axis=1)

# # === 4. 反归一化 ===训练集反归一化
# y_test_inv1 = scaler_y.inverse_transform(y_train)
# y_pred_inv1 = scaler_y.inverse_transform(y_pred1)

# # === 5. 绘图 & 分目标 R² ===只画test集的
# # fig, axs = plt.subplots(3, 1, figsize=(10, 12))
# r2_scores = []

# for i in range(3):
#     r2 = r2_score(y_test_inv[:, i], y_pred_inv[:, i])
#     r2_scores.append(r2)
#     MSE=mean_squared_error(y_test_inv[:, i], y_pred_inv[:, i])
#     # axs[i].plot(y_test_inv[:, i], label='True', color='blue', marker='o', linestyle='-')
#     # axs[i].plot(y_pred_inv[:, i], label='Predicted', color='red', marker='x', linestyle='--')
#     # axs[i].set_title(f'目标 {i+1} 的预测结果（原始尺度） - R² = {r2:.4f},MSE={MSE:.4f}')
#     # axs[i].legend()
#     # axs[i].grid(True)
# print('测试集的R2：',r2_scores)
# # plt.tight_layout()
# # plt.show()
# # === 5. 绘图 & 分目标 R² ===只画test集的
# # fig, axs = plt.subplots(3, 1, figsize=(10, 12))
# r2_scores = []

# for i in range(3):
#     r2 = r2_score(y_test_inv1[:, i], y_pred_inv1[:, i])
#     r2_scores.append(r2)
#     MSE=mean_squared_error(y_test_inv1[:, i], y_pred_inv1[:, i])
#     # axs[i].plot(y_test_inv[:, i], label='True', color='blue', marker='o', linestyle='-')
#     # axs[i].plot(y_pred_inv[:, i], label='Predicted', color='red', marker='x', linestyle='--')
#     # axs[i].set_title(f'目标 {i+1} 的预测结果（原始尺度） - R² = {r2:.4f},MSE={MSE:.4f}')
#     # axs[i].legend()
#     # axs[i].grid(True)
# print('训练集的R2：',r2_scores)
# # plt.tight_layout()
# # plt.show()
# # === 4. 整体评估指标 ===
# overall_r2 = r2_score(y_test_inv, y_pred_inv)
# overall_mse = mean_squared_error(y_test_inv, y_pred_inv)
# overall_mae = mean_absolute_error(y_test_inv, y_pred_inv)

# print("========== 总体评估指标 ==========")
# print(f"整体 R²:  {overall_r2:.4f}")
# print(f"整体 MSE: {overall_mse:.4f}")
# print(f"整体 MAE: {overall_mae:.4f}")
# print("=================================")



# # 将反归一化后的数据转为 DataFrame
# df_test = pd.DataFrame(y_test_inv, columns=['emod', 'krat', 'demod'])
# df_pred = pd.DataFrame(y_pred_inv, columns=['emod_pred', 'krat_pred', 'demod_pred'])

# # 合并两个表（列并排）
# df_result = pd.concat([df_test, df_pred], axis=1)

# # 保存为 Excel 文件
# df_result.to_excel('ANNtest_result.xlsx', index=False)
# df_test = pd.DataFrame(y_test_inv, columns=['emod', 'krat', 'demod'])
# df_pred = pd.DataFrame(y_pred_inv, columns=['emod_pred', 'krat_pred', 'demod_pred'])


# # 将反归一化后的数据转为 DataFrame
# df_test1 = pd.DataFrame(y_test_inv1, columns=['emod', 'krat', 'demod'])
# df_pred1 = pd.DataFrame(y_pred_inv1, columns=['emod_pred', 'krat_pred', 'demod_pred'])

# # 合并两个表（列并排）
# df_result = pd.concat([df_test1, df_pred1], axis=1)

# # 保存为 Excel 文件
# df_result.to_excel('ANNtrain_results.xlsx', index=False)
# df_test = pd.DataFrame(y_test_inv1, columns=['emod', 'krat', 'demod'])
# df_pred = pd.DataFrame(y_pred_inv1, columns=['emod_pred', 'krat_pred', 'demod_pred'])


RobustScaler 已保存为 y_scaler.save
X: [[0.30105372 0.03512799 0.29436349 ... 0.08625962 0.99484596 0.00418607]
 [0.05748919 0.01870521 0.05564587 ... 0.26603876 0.99001528 0.0020889 ]
 [0.04468457 0.00646271 0.0433707  ... 0.06754668 0.9980459  0.01514082]
 ...
 [0.06050246 0.01610081 0.05872733 ... 0.21570299 0.99943575 0.46153358]
 [0.21918189 0.07057651 0.21383195 ... 0.26283177 0.98491386 0.01097889]
 [0.11735513 0.0327999  0.11416508 ... 0.2279874  0.98803705 0.0060761 ]] X_else: []
X_train min: 0.0 max: 1.0
y_scaled min: -1.1333333333333333 max: 5.466666666666666
Epoch 1/1000


C:\Users\14397\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - loss: 0.4275 - r_squared: -0.0713 - val_loss: 0.3837 - val_r_squared: -0.0693
Epoch 2/1000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.4584 - r_squared: -0.0400 - val_loss: 0.3740 - val_r_squared: -0.0506
Epoch 3/1000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.4334 - r_squared: -0.0204 - val_loss: 0.3631 - val_r_squared: -0.0342
Epoch 4/1000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.4434 - r_squared: 0.0061 - val_loss: 0.3496 - val_r_squared: -0.0140
Epoch 5/1000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3433 - r_squared: -0.0289 - val_loss: 0.3400 - val_r_squared: 0.0043
Epoch 6/1000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3911 - r_squared: 0.0316 - val_loss: 0.3324 - val_r_squared: 0.0176
Epoch 7/1000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.3579 - r_squared: 0.0395 - val_loss: 0.3188 - val_r_squared: 0.0293
Epoch 8/1000
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.3699 - r_squared: 0.0682 - val_loss: 0.31